# SupplyForge AI 2.0: KAN-35 - Hierarchical Aggregation Features



Tài liệu này thực hiện việc tính toán các đặc trưng chuỗi thời gian phân cấp (Hierarchical Time-Series Features) ở cấp độ Nhóm sản phẩm (Brand và Category) để hỗ trợ dự báo nhu cầu SKU có dữ liệu thưa thớt hoặc SKU mới (Cold-Start).



Các đặc trưng bao gồm lags (1, 7 ngày), rolling mean (7, 30 ngày), và doanh thu lũy kế (cumulative sales) ở cả cấp Brand và Category. Các đặc trưng này được bảo vệ chống rò rỉ dữ liệu (Target Leakage) bằng cách lùi ngày `shift(1)` trước mọi phép tính rolling và cumulative.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Bảo toàn số lượng dòng**: Kích thước lưới chuỗi thời gian không thay đổi (giữ nguyên 323,250 dòng).

2. **Chống rò rỉ dữ liệu**: Chỉ sử dụng lịch sử lùi ngày (`shift(1)`) của nhóm làm đầu vào cho rolling và cumulative.

3. **Tách biệt phân cấp**: Các phép tính đặc trưng được thực hiện độc lập cho từng nhóm phân cấp trước khi left join vào lưới.


In [1]:
import pandas as pd

import numpy as np

import os



PROJECT_ROOT = '/home/nguyen-dai-tung/supplyforge-ai'

GOLD_DIR = os.path.join(PROJECT_ROOT, 'data', 'gold')

grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.parquet')

if not os.path.exists(grid_path):

    grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải lưới dữ liệu từ Gold Layer...")

df_grid = pd.read_parquet(grid_path) if grid_path.endswith(".parquet") else pd.read_csv(grid_path)

print(f"Tổng số dòng: {len(df_grid):,}") 


Đang tải lưới dữ liệu từ Gold Layer...
Tổng số dòng: 1,597,060


In [2]:
def generate_hierarchical_features(df):
    df_out = df.sort_values(["brand", "categoryId", "itemId", "date"]).copy()
    
    # Brand level
    brand_daily = df_out.groupby(["brand", "date"])["quantity"].sum().reset_index()
    brand_daily = brand_daily.sort_values(["brand", "date"])
    brand_daily.rename(columns={"quantity": "brand_daily_sales"}, inplace=True)
    brand_daily["brand_lag_1"] = brand_daily.groupby("brand")["brand_daily_sales"].shift(1).astype("float32")
    brand_daily["brand_roll_mean_7"] = brand_daily.groupby("brand")["brand_daily_sales"].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()).astype("float32")
    brand_daily["brand_cumulative_sales"] = brand_daily.groupby("brand")["brand_daily_sales"].transform(lambda x: x.shift(1).cumsum()).fillna(0).astype("float32")
    brand_daily["brand_daily_sales"] = brand_daily["brand_daily_sales"].astype("float32")
    
    # Category level
    cat_daily = df_out.groupby(["categoryId", "date"])["quantity"].sum().reset_index()
    cat_daily = cat_daily.sort_values(["categoryId", "date"])
    cat_daily.rename(columns={"quantity": "category_daily_sales"}, inplace=True)
    cat_daily["category_lag_1"] = cat_daily.groupby("categoryId")["category_daily_sales"].shift(1).astype("float32")
    cat_daily["category_roll_mean_7"] = cat_daily.groupby("categoryId")["category_daily_sales"].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()).astype("float32")
    cat_daily["category_cumulative_sales"] = cat_daily.groupby("categoryId")["category_daily_sales"].transform(lambda x: x.shift(1).cumsum()).fillna(0).astype("float32")
    cat_daily["category_daily_sales"] = cat_daily["category_daily_sales"].astype("float32")
    
    df_out = pd.merge(df_out, brand_daily, on=["brand", "date"], how="left")
    df_out = pd.merge(df_out, cat_daily, on=["categoryId", "date"], how="left")
    return df_out.sort_values(["itemId", "date"]).reset_index(drop=True)

print("Đang chạy quy trình tính toán đặc trưng phân cấp...")
df_featured = generate_hierarchical_features(df_grid)
print(f"Kích thước lưới sau khi tính toán: {df_featured.shape}")
print("Danh sách đặc trưng phân cấp mới thêm vào:")
print(list(df_featured.columns))


Đang chạy quy trình tính toán đặc trưng phân cấp...


Kích thước lưới sau khi tính toán: (1597060, 37)
Danh sách đặc trưng phân cấp mới thêm vào:
['date', 'itemId', 'quantity', 'brand', 'categoryId', 'price', 'cost', 'warehouse', 'brand_encoded', 'categoryId_encoded', 'warehouse_encoded', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_sum_7', 'rolling_mean_7', 'rolling_std_7', 'zero_sales_ratio_7', 'rolling_sum_14', 'rolling_mean_14', 'rolling_std_14', 'zero_sales_ratio_14', 'rolling_sum_28', 'rolling_mean_28', 'rolling_std_28', 'zero_sales_ratio_28', 'ema_7', 'days_since_last_sale', 'brand_daily_sales', 'brand_lag_1', 'brand_roll_mean_7', 'brand_cumulative_sales', 'category_daily_sales', 'category_lag_1', 'category_roll_mean_7', 'category_cumulative_sales']


In [3]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")







# 1. Xác thực số lượng dòng được bảo toàn



print(f"Số dòng trước: {len(df_grid)} | Số dòng sau: {len(df_featured)}") 


assert len(df_grid) == len(df_featured), "Lỗi: Số lượng dòng lưới bị thay đổi!"



print("1. Đảm bảo bảo toàn số dòng lưới: Đạt")







# 2. Kiểm chứng thủ công toán học trên một Brand và một Category mẫu



sample_brand = df_featured['brand'].dropna().unique()[0]



sample_cat = df_featured['categoryId'].dropna().unique()[0]







print(f"Kiểm chứng thủ công Brand: {sample_brand}") 


# Tính thực tế từ dữ liệu đầu vào của Brand



df_brand_actual = df_grid[df_grid['brand'] == sample_brand].groupby('date')['quantity'].sum().reset_index()



df_brand_actual = df_brand_actual.sort_values('date').reset_index(drop=True)







# Lấy kết quả đặc trưng từ dataframe sau tính toán (1 dòng đại diện mỗi ngày)



df_brand_featured = df_featured[df_featured['brand'] == sample_brand].drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)







test_idx = 42



qty_history = df_brand_actual['quantity'].iloc[:test_idx].values







expected_lag1 = qty_history[-1]



expected_sum7 = sum(qty_history[-7:])



expected_mean7 = expected_sum7 / 7.0



expected_cumulative = sum(qty_history)







actual_lag1 = df_brand_featured['brand_lag_1'].iloc[test_idx]



actual_mean7 = df_brand_featured['brand_roll_mean_7'].iloc[test_idx]



actual_cumulative = df_brand_featured['brand_cumulative_sales'].iloc[test_idx]







print(f" - lag_1: Thực tế={actual_lag1}, Kỳ vọng={expected_lag1}") 


print(f" - roll_mean_7: Thực tế={actual_mean7}, Kỳ vọng={expected_mean7}") 


print(f" - cumulative_sales: Thực tế={actual_cumulative}, Kỳ vọng={expected_cumulative}") 






assert abs(actual_lag1 - expected_lag1) < 1e-5, "Lỗi đối chiếu brand_lag_1!"



assert abs(actual_mean7 - expected_mean7) < 1e-5, "Lỗi đối chiếu brand_roll_mean_7!"



assert abs(actual_cumulative - expected_cumulative) < 1e-5, "Lỗi đối chiếu brand_cumulative_sales!"



print("2. Đối chiếu kiểm chứng mẫu Brand hoàn toàn chính xác: Đạt\n")







print(f"Kiểm chứng thủ công Category: {sample_cat}") 


df_cat_actual = df_grid[df_grid['categoryId'] == sample_cat].groupby('date')['quantity'].sum().reset_index()



df_cat_actual = df_cat_actual.sort_values('date').reset_index(drop=True)







df_cat_featured = df_featured[df_featured['categoryId'] == sample_cat].drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)







qty_cat_history = df_cat_actual['quantity'].iloc[:test_idx].values







expected_cat_lag1 = qty_cat_history[-1]



expected_cat_sum7 = sum(qty_cat_history[-7:])



expected_cat_mean7 = expected_cat_sum7 / 7.0



expected_cat_cumulative = sum(qty_cat_history)







actual_cat_lag1 = df_cat_featured['category_lag_1'].iloc[test_idx]



actual_cat_mean7 = df_cat_featured['category_roll_mean_7'].iloc[test_idx]



actual_cat_cumulative = df_cat_featured['category_cumulative_sales'].iloc[test_idx]







print(f" - lag_1: Thực tế={actual_cat_lag1}, Kỳ vọng={expected_cat_lag1}") 


print(f" - roll_mean_7: Thực tế={actual_cat_mean7}, Kỳ vọng={expected_cat_mean7}") 


print(f" - cumulative_sales: Thực tế={actual_cat_cumulative}, Kỳ vọng={expected_cat_cumulative}") 






assert abs(actual_cat_lag1 - expected_cat_lag1) < 1e-5, "Lỗi đối chiếu category_lag_1!"



assert abs(actual_cat_mean7 - expected_cat_mean7) < 1e-5, "Lỗi đối chiếu category_roll_mean_7!"



assert abs(actual_cat_cumulative - expected_cat_cumulative) < 1e-5, "Lỗi đối chiếu category_cumulative_sales!"



print("3. Đối chiếu kiểm chứng mẫu Category hoàn toàn chính xác: Đạt\n")



=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

Số dòng trước: 1597060 | Số dòng sau: 1597060
1. Đảm bảo bảo toàn số dòng lưới: Đạt
Kiểm chứng thủ công Brand: NoBrand
 - lag_1: Thực tế=0.0, Kỳ vọng=0
 - roll_mean_7: Thực tế=0.0, Kỳ vọng=0.0
 - cumulative_sales: Thực tế=25.0, Kỳ vọng=25
2. Đối chiếu kiểm chứng mẫu Brand hoàn toàn chính xác: Đạt

Kiểm chứng thủ công Category: NoCategory


 - lag_1: Thực tế=4.0, Kỳ vọng=4
 - roll_mean_7: Thực tế=3.857142925262451, Kỳ vọng=3.857142857142857
 - cumulative_sales: Thực tế=288.0, Kỳ vọng=288
3. Đối chiếu kiểm chứng mẫu Category hoàn toàn chính xác: Đạt



In [4]:
output_parquet = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.parquet")
output_csv = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.csv")
df_featured.to_parquet(output_parquet, index=False)
df_featured.to_csv(output_csv, index=False)
print(f"Đã lưu đặc trưng phân cấp tại Gold Layer: {output_parquet}")


Đã lưu đặc trưng phân cấp tại Gold Layer: /home/nguyen-dai-tung/supplyforge-ai/data/gold/calendar_daily_sales_metadata.parquet
